In [ ]:
### Step 1 — Setup

import sys, os
sys.path.insert(0, os.path.expanduser("~/Sleep_Stage_Research/conference"))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader

import config as cfg
from data_utils import (compute_class_weights, compute_channel_stats,
                        load_subjects_from_h5, load_participant_info)
from dataset import SleepSequenceDataset
from model import SleepStageNet, count_parameters
from trainer import Trainer
from eval_utils import (predict_on_subjects, compute_fold_metrics, aggregate_cv_results,
                        print_metrics, print_cv_summary, plot_confusion_matrix,
                        plot_training_history, save_results)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

h5_path = os.path.join(cfg.PREPROCESSED_DIR, "dreamt_psg7ch_epochs.h5")
pinfo = load_participant_info()

# Age groups
def assign_age_group(age):
    if age < 50: return "<50"
    elif age <= 65: return "50-65"
    else: return ">65"

pinfo["age_group"] = pinfo["AGE"].apply(assign_age_group)
age_groups = {}
for grp in ["<50", "50-65", ">65"]:
    subjects = sorted(pinfo[pinfo["age_group"] == grp]["SID"].tolist())
    age_groups[grp] = subjects
    print(f"Age {grp}: n={len(subjects)}")

# Load age fold assignments
folds_path = os.path.join(cfg.CHECKPOINT_DIR, "exp3_age_fold_assignments.json")
with open(folds_path) as f:
    age_folds_raw = json.load(f)
age_folds = {}
for g in ["<50", "50-65", ">65"]:
    age_folds[g] = {int(k): v for k, v in age_folds_raw[g].items()}
print(f"[LOAD] Age fold assignments (same splits as Exp 3)")

# Universal fold assignments
with open(os.path.join(cfg.CHECKPOINT_DIR, "fold_assignments.json")) as f:
    universal_folds = json.load(f)

# Load Exp 0 and Exp 3 per-subject results
exp0_per_subj = {}
for fold_idx in range(cfg.NUM_FOLDS):
    rpath = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "test_results.json")
    if os.path.exists(rpath):
        with open(rpath) as f:
            for sid, m in json.load(f)["per_subject"].items():
                exp0_per_subj[sid] = m

exp3_per_subj = {}
for grp in ["<50", "50-65", ">65"]:
    grp_tag = grp.replace("<", "lt").replace(">", "gt").replace("-", "_")
    for prefix in [f"exp3_psg_{grp_tag}", f"exp3_{grp_tag}"]:
        for fold_idx in range(cfg.NUM_FOLDS):
            rpath = os.path.join(cfg.CHECKPOINT_DIR, f"{prefix}_fold{fold_idx}", "test_results.json")
            if os.path.exists(rpath):
                with open(rpath) as f:
                    for sid, m in json.load(f)["per_subject"].items():
                        exp3_per_subj[sid] = m

print(f"Loaded Exp 0 results: {len(exp0_per_subj)} subjects")
print(f"Loaded Exp 3 results: {len(exp3_per_subj)} subjects")

FT_LR = 1e-4
FT_MAX_EPOCHS = 20
FT_EARLY_STOP = 7
FT_LR_PATIENCE = 3
EXP_NAME = "exp3_1_psg"
RNN_MODE = "bilstm"

print(f"\nFine-tuning config: lr={FT_LR}, max_epochs={FT_MAX_EPOCHS}, early_stop={FT_EARLY_STOP}")

In [ ]:
### Step 2 — Fine-tuning trainer

class FineTuneTrainer(Trainer):
    def __init__(self, model, train_loader, val_loader, class_weights,
                 pretrained_path, exp_name, fold, device):
        super().__init__(model, train_loader, val_loader, class_weights,
                         exp_name=exp_name, fold=fold, device=device)
        self.optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", patience=FT_LR_PATIENCE, factor=cfg.LR_FACTOR)
        self.pretrained_path = pretrained_path

    def train(self):
        if self.is_fold_complete():
            history_path = os.path.join(self.save_dir, "train_history.json")
            if os.path.exists(history_path):
                with open(history_path, "r") as f:
                    self.history = json.load(f)
            print(f"  [SKIP] {self.exp_name} fold {self.fold} already complete.")
            return self.history

        resumed = self.load_checkpoint()
        if not resumed:
            print(f"  [INIT] Loading pre-trained weights from {self.pretrained_path}")
            ckpt = torch.load(self.pretrained_path, map_location=self.device, weights_only=False)
            state = ckpt["model_state"]
            remapped = {}
            for k, v in state.items():
                new_k = k.replace("lstm.", "rnn.") if k.startswith("lstm.") else k
                remapped[new_k] = v
            self.model.load_state_dict(remapped)
            print(f"  [INIT] Pre-trained model loaded (val_loss={ckpt['best_val_loss']:.4f})")

        import time
        for epoch in range(self.start_epoch, FT_MAX_EPOCHS):
            t0 = time.time()
            current_lr = self.optimizer.param_groups[0]["lr"]
            print(f"\n  Epoch {epoch+1}/{FT_MAX_EPOCHS} | lr={current_lr:.2e}")

            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate()
            self.scheduler.step(val_loss)

            elapsed = time.time() - t0
            print(f"  => train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | {elapsed:.0f}s")

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)
            self.history["lr"].append(current_lr)

            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                print(f"  ** New best val_loss: {val_loss:.4f}")
            else:
                self.patience_counter += 1
                print(f"  Patience: {self.patience_counter}/{FT_EARLY_STOP}")

            self.save_checkpoint(epoch, is_best=is_best)
            if self.patience_counter >= FT_EARLY_STOP:
                print(f"  [EARLY STOP] No improvement for {FT_EARLY_STOP} epochs.")
                break

        self.mark_complete()
        print(f"  [DONE] {self.exp_name} fold {self.fold} complete. Best val_loss={self.best_val_loss:.4f}")
        with open(os.path.join(self.save_dir, "train_history.json"), "w") as f:
            json.dump(self.history, f, indent=2)
        return self.history

print("FineTuneTrainer ready.")

In [ ]:
### Step 3 — Find universal checkpoint helper + Train all age groups

def find_universal_checkpoint(test_subjects):
    min_overlap = len(test_subjects)
    best_fold = 0
    for fold_idx in range(cfg.NUM_FOLDS):
        exp0_test = set(universal_folds[str(fold_idx)]["test"])
        overlap = len(set(test_subjects) & exp0_test)
        if overlap == 0:
            path = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "best_model.pt")
            if os.path.exists(path):
                return path, fold_idx
        if overlap < min_overlap:
            min_overlap = overlap
            best_fold = fold_idx
    return os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{best_fold}", "best_model.pt"), best_fold


all_results = {}
all_histories = {}

for grp in ["<50", "50-65", ">65"]:
    grp_tag = grp.replace("<", "lt").replace(">", "gt").replace("-", "_")
    folds = age_folds[grp]

    all_results[grp] = []
    all_histories[grp] = []

    for fold_idx in range(cfg.NUM_FOLDS):
        exp_tag = f"{EXP_NAME}_{grp_tag}_fold{fold_idx}"

        print(f"\n{'#'*70}")
        print(f"# FINE-TUNE Age {grp} | FOLD {fold_idx+1}/{cfg.NUM_FOLDS}")
        print(f"{'#'*70}")

        train_subjects = folds[fold_idx]["train"]
        test_subjects = folds[fold_idx]["test"]

        np.random.seed(cfg.SEED + fold_idx)
        n_val = max(1, int(len(train_subjects) * cfg.VAL_RATIO))
        perm = np.random.permutation(len(train_subjects))
        val_subjects = [train_subjects[i] for i in perm[:n_val]]
        actual_train = [train_subjects[i] for i in perm[n_val:]]

        print(f"  Train: {len(actual_train)} | Val: {len(val_subjects)} | Test: {len(test_subjects)}")

        pretrained_path, univ_fold = find_universal_checkpoint(test_subjects)
        print(f"  Pre-trained from: exp0_fold{univ_fold}")

        stats_path = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{univ_fold}", "channel_stats.npz")
        stats = np.load(stats_path)
        mean, std = stats["mean"], stats["std"]

        ft_stats_path = os.path.join(cfg.CHECKPOINT_DIR, exp_tag, "channel_stats.npz")
        os.makedirs(os.path.dirname(ft_stats_path), exist_ok=True)
        np.savez(ft_stats_path, mean=mean, std=std)

        _, train_labels, _ = load_subjects_from_h5(h5_path, actual_train)
        class_weights = compute_class_weights(train_labels)
        del train_labels

        train_ds = SleepSequenceDataset(h5_path, actual_train, mean=mean, std=std)
        val_ds = SleepSequenceDataset(h5_path, val_subjects, mean=mean, std=std)
        print(f"  Train sequences: {len(train_ds)}, Val sequences: {len(val_ds)}")

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                                num_workers=0, pin_memory=True)

        model = SleepStageNet(n_channels=len(cfg.PSG_CHANNELS), rnn_mode=RNN_MODE).to(device)
        trainer = FineTuneTrainer(model, train_loader, val_loader, class_weights,
                                  pretrained_path=pretrained_path,
                                  exp_name=f"{EXP_NAME}_{grp_tag}", fold=fold_idx, device=device)

        history = trainer.train()
        all_histories[grp].append(history)

        print(f"\n  [EVAL] Evaluating on {len(test_subjects)} test subjects...")
        trainer.load_best_model()
        results = predict_on_subjects(model, h5_path, test_subjects, mean, std, device=device)
        overall, per_subj = compute_fold_metrics(results)
        print_metrics(overall, title=f"FT-Age{grp} Fold {fold_idx}")
        all_results[grp].append((overall, per_subj))

        fold_rpath = os.path.join(cfg.CHECKPOINT_DIR, exp_tag, "test_results.json")
        save_results({"overall": overall, "per_subject": {s: m for s, m in per_subj.items()}}, fold_rpath)

        del model, trainer, train_ds, val_ds, train_loader, val_loader
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*70}")
print(f"EXP 3.1: ALL AGE FINE-TUNING COMPLETE")
print(f"{'='*70}")

In [ ]:
### Step 4 — CV summaries

age_summaries = {}
age_per_subj = {}

for grp in ["<50", "50-65", ">65"]:
    grp_tag = grp.replace("<", "lt").replace(">", "gt").replace("-", "_")
    summary = aggregate_cv_results(all_results[grp])
    age_summaries[grp] = summary

    print(f"\n{'='*60}")
    print(f"  FT-Age {grp} Model CV Summary")
    print(f"{'='*60}")
    print_cv_summary(summary)
    save_results(summary, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_{grp_tag}_cv_summary.json"))

    per_subj_all = {}
    for overall, per_subj in all_results[grp]:
        per_subj_all.update(per_subj)
    age_per_subj[grp] = per_subj_all

In [ ]:
### Step 5 — Three-way comparison: Universal vs From-Scratch vs Fine-Tuned

from scipy.stats import wilcoxon

print(f"\n{'='*70}")
print(f"  THREE-WAY COMPARISON: Universal vs From-Scratch vs Fine-Tuned (Age)")
print(f"{'='*70}")

comparison_rows = []

for grp in ["<50", "50-65", ">65"]:
    subjects = age_groups[grp]

    u_kappas = [exp0_per_subj[s]["kappa"] for s in subjects if s in exp0_per_subj]
    s_kappas = [exp3_per_subj[s]["kappa"] for s in subjects if s in exp3_per_subj]
    ft_kappas = [age_per_subj[grp][s]["kappa"] for s in subjects if s in age_per_subj[grp]]

    u_accs = [exp0_per_subj[s]["accuracy"] for s in subjects if s in exp0_per_subj]
    s_accs = [exp3_per_subj[s]["accuracy"] for s in subjects if s in exp3_per_subj]
    ft_accs = [age_per_subj[grp][s]["accuracy"] for s in subjects if s in age_per_subj[grp]]

    u_f1s = [exp0_per_subj[s]["f1_macro"] for s in subjects if s in exp0_per_subj]
    s_f1s = [exp3_per_subj[s]["f1_macro"] for s in subjects if s in exp3_per_subj]
    ft_f1s = [age_per_subj[grp][s]["f1_macro"] for s in subjects if s in age_per_subj[grp]]

    print(f"\n  Age {grp} (n={len(subjects)}):")
    print(f"  {'Metric':<15} {'Universal':>18} {'From-Scratch':>18} {'Fine-Tuned':>18}")
    print(f"  {'-'*69}")

    for mname, u_v, s_v, ft_v in [
        ("Accuracy", u_accs, s_accs, ft_accs),
        ("F1 (macro)", u_f1s, s_f1s, ft_f1s),
        ("Kappa", u_kappas, s_kappas, ft_kappas),
    ]:
        print(f"  {mname:<15} {np.mean(u_v):>14.4f}     {np.mean(s_v):>14.4f}     {np.mean(ft_v):>14.4f}")

    common = [s for s in subjects if s in exp0_per_subj and s in age_per_subj[grp]]
    if len(common) > 5:
        u_k = [exp0_per_subj[s]["kappa"] for s in common]
        ft_k = [age_per_subj[grp][s]["kappa"] for s in common]
        stat, p = wilcoxon(u_k, ft_k)
        sig = "YES" if p < 0.05 else "NO"
        print(f"  Wilcoxon (Universal vs Fine-Tuned): p={p:.4f} -> Significant: {sig}")

    common2 = [s for s in subjects if s in exp3_per_subj and s in age_per_subj[grp]]
    if len(common2) > 5:
        s_k2 = [exp3_per_subj[s]["kappa"] for s in common2]
        ft_k2 = [age_per_subj[grp][s]["kappa"] for s in common2]
        stat2, p2 = wilcoxon(s_k2, ft_k2)
        sig2 = "YES" if p2 < 0.05 else "NO"
        print(f"  Wilcoxon (From-Scratch vs Fine-Tuned): p={p2:.4f} -> Significant: {sig2}")

    comparison_rows.append({
        "age_group": grp,
        "universal_kappa": float(np.mean(u_kappas)),
        "scratch_kappa": float(np.mean(s_kappas)),
        "finetune_kappa": float(np.mean(ft_kappas)),
    })

save_results(comparison_rows, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_comparison.json"))

In [ ]:
### Step 6 — Visualizations

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"font.size": 11})

for grp in ["<50", "50-65", ">65"]:
    grp_tag = grp.replace("<", "lt").replace(">", "gt").replace("-", "_")
    for fold_idx, history in enumerate(all_histories[grp]):
        if history and history.get("train_loss"):
            plot_training_history(
                history, title=f"FT-Age{grp} Fold {fold_idx}",
                save_path=os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_{grp_tag}_fold{fold_idx}", "training_curves.png"))

for grp in ["<50", "50-65", ">65"]:
    grp_tag = grp.replace("<", "lt").replace(">", "gt").replace("-", "_")
    cm_total = np.zeros((cfg.NUM_CLASSES, cfg.NUM_CLASSES), dtype=np.int64)
    for overall, _ in all_results[grp]:
        cm_total += np.array(overall["confusion_matrix"])
    plot_confusion_matrix(cm_total, title=f"Exp 3.1: FT-Age{grp}",
                          save_path=os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_{grp_tag}_confusion_matrix.png"))

# Three-way bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
grp_labels = ["<50", "50-65", ">65"]

for ax, metric, mlabel in zip(axes, ["accuracy", "f1_macro", "kappa"], ["Accuracy", "F1 (Macro)", "Kappa"]):
    x = np.arange(len(grp_labels))
    width = 0.25

    u_vals, s_vals, ft_vals = [], [], []
    for grp in grp_labels:
        subjects = age_groups[grp]
        u_vals.append(np.mean([exp0_per_subj[s][metric] for s in subjects if s in exp0_per_subj]))
        s_vals.append(np.mean([exp3_per_subj[s][metric] for s in subjects if s in exp3_per_subj]))
        ft_vals.append(np.mean([age_per_subj[grp][s][metric] for s in subjects if s in age_per_subj[grp]]))

    bars1 = ax.bar(x - width, u_vals, width, label="Universal", color="#3498db", edgecolor="black", linewidth=0.5)
    bars2 = ax.bar(x, s_vals, width, label="From-Scratch", color="#e74c3c", edgecolor="black", linewidth=0.5)
    bars3 = ax.bar(x + width, ft_vals, width, label="Fine-Tuned", color="#2ecc71", edgecolor="black", linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(grp_labels)
    ax.set_xlabel("Age Group")
    ax.set_ylabel(mlabel)
    ax.set_title(mlabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f"{bar.get_height():.3f}", ha="center", fontsize=7)

plt.suptitle("Exp 3.1: Universal vs From-Scratch vs Fine-Tuned (Age Groups)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_threeway_chart.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nExperiment 3.1 complete.")